# Week 6, Day 5: Docstrings

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w006d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def f(a, b=2):
    return a ** b

print(f(3), f(2, 5))

In [ ]:
def power(base, exponent=2):
    """Return base raised to the power exponent (squared by default)."""
    return base ** exponent

print(power.__doc__)
help(power)

In [ ]:
def power(base, exponent=2):
    result = base ** exponent
    """Return base raised to the power exponent."""
    return result

print(power.__doc__)

> **Key idea.** A docstring is a string literal, usually in triple quotes, on the first line of a function's body. It describes **what** the function does and **how to use it**, not how it works inside.

In [ ]:
def money(amount, symbol="₦", places=2):
    """Format an amount of money for display.

    amount: the amount, as an int or a float.
    symbol: the currency symbol to put in front.
    places: how many decimal places to show.

    Returns a string with a thousands separator, such as "₦1,234.50".
    Negative amounts get the minus sign before the symbol: "-₦5.00".
    """
    text = f"{symbol}{abs(amount):,.{places}f}"
    return "-" + text if amount < 0 else text

help(money)
print(money(-5))

> **Tip.** A good test for a docstring: could someone use the function correctly after reading only the `def` line and the docstring? If they'd still need to read the code, the docstring isn't finished.

In [ ]:
def digit_sum(n):
    """Return the sum of the decimal digits of n.

    n must be a whole number of at least 0.
    """
    total = 0
    while n > 0:
        total += n % 10
        n //= 10
    return total

print(digit_sum(4096))

In [ ]:
import doctest

def initials(name):
    """Return the initials of a name, in capitals with full stops.

    >>> initials("ada lovelace")
    'A.L.'
    >>> initials("Grace Brewster Hopper")
    'G.B.H.'
    """
    result = ""
    for part in name.split():
        result += part[0].upper() + "."
    return result

print(doctest.run_docstring_examples(initials, {"initials": initials}, verbose=False))
print("Examples checked")

> **Key idea.** Examples in a docstring do two jobs: they show people how to use the function, and with `doctest` they prove the documentation is still true.

In [ ]:
def is_leap(year):
    """Return True if year is a leap year in the Gregorian calendar."""
    # Century years are only leap years when they divide by 400:
    # 2000 was a leap year, 1900 was not.
    return (year % 4 == 0 and year % 100 != 0) or year % 400 == 0

> **Common mistake.** Comments that repeat the code add nothing: `i += 1  # add one to i`. Comment the **why**, not the **what**.

### Exercise: Document them

Add a one-line docstring to each function, saying what it returns. Each must be a command ending with a full stop, like `"""Return ..."""`.

In [ ]:
def area(width, height):
    return width * height

def is_vowel(ch):
    return ch.lower() in "aeiou"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_docs():\n    \"both have docstrings\"\n    for f in [area, is_vowel]:\n        d = f.__doc__\n        assert d, f.__name__ + \" has no docstring. It must be the first line of the body.\"\n        first = d.strip().splitlines()[0]\n        assert first.startswith(\"Return\") and first.endswith(\".\"), f\"Start {f.__name__}'s summary with 'Return' and end it with a full stop: {first!r}\"\ndef test_still_work():\n    \"the functions still work\"\n    assert area(3, 4) == 12 and is_vowel(\"E\") and not is_vowel(\"x\")")

### Exercise: Write it from its docstring

Here is a docstring with no body. Write the body so that the function keeps every promise in the docstring.

In [ ]:
def clamp(value, low, high):
    """Return value, limited to the range low to high.

    If value is below low, return low. If it is above high, return high.
    Otherwise return value unchanged. low must not be greater than high.

    >>> clamp(5, 0, 10)
    5
    >>> clamp(-3, 0, 10)
    0
    >>> clamp(99, 0, 10)
    10
    """
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"keeps the contract\"\n    for v, lo, hi, want in [(5, 0, 10, 5), (-3, 0, 10, 0), (99, 0, 10, 10), (0, 0, 10, 0), (10, 0, 10, 10), (7, 7, 7, 7)]:\n        assert clamp(v, lo, hi) == want, f\"clamp({v}, {lo}, {hi}) should be {want}.\"\ndef test_doctest():\n    \"the docstring's examples pass\"\n    import doctest\n    r = doctest.run_docstring_examples(clamp, {\"clamp\": clamp}, verbose=False)\n    assert clamp.__doc__ and \">>> clamp(5, 0, 10)\" in clamp.__doc__, \"Keep the docstring as it is.\"")

### Exercise: Write the examples

`reverse_words` works, but its docstring has no examples. Add **at least two** doctest examples to its docstring (lines starting with `>>> `, each followed by the exact result), one of them with extra spaces in the input. The checks will run your examples, so they must be right.

In [ ]:
def reverse_words(sentence):
    """Return the words of sentence in reverse order, separated by single spaces.
    """
    words = sentence.split()
    result = ""
    for w in words:
        result = w if result == "" else w + " " + result
    return result

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_examples():\n    \"has at least two examples, and they pass\"\n    import doctest\n    doc = reverse_words.__doc__ or \"\"\n    assert doc.count(\">>> \") >= 2, \"Add at least two examples, each on a line starting with >>> .\"\n    finder = doctest.DocTestFinder()\n    runner = doctest.DocTestRunner(verbose=False)\n    for t in finder.find(reverse_words, \"reverse_words\", globs={\"reverse_words\": reverse_words}):\n        runner.run(t, out=lambda s: None)\n    assert runner.failures == 0, \"One of your examples doesn't match what the function returns. Strings are shown with quotes.\"\ndef test_spaces():\n    \"one example has extra spaces\"\n    import re\n    calls = re.findall(r\">>> reverse_words\\((.*)\\)\", reverse_words.__doc__ or \"\")\n    assert any(\"  \" in c for c in calls), \"Make one example use extra spaces, like '  a   b '.\"")

### Exercise: State the contract

Write a docstring for `average_of` that has a summary line, then a line starting `numbers:` describing what the argument must be (a string of whole numbers separated by spaces, with at least one number), and a line starting `Returns` describing the result. Then write the body.

In [ ]:
def average_of(numbers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_doc():\n    \"the docstring has the three parts\"\n    d = average_of.__doc__ or \"\"\n    lines = [l.strip() for l in d.strip().splitlines()]\n    assert lines and lines[0].endswith(\".\"), \"Start with a one-line summary ending in a full stop.\"\n    assert any(l.startswith(\"numbers:\") for l in lines), \"Add a line starting 'numbers:'.\"\n    assert any(l.startswith(\"Returns\") for l in lines), \"Add a line starting 'Returns'.\"\ndef test_values():\n    \"the function works\"\n    assert average_of(\"4 8 15 16 23 42\") == 18.0\n    assert average_of(\"7\") == 7.0")

### Exercise: A comment that explains why

This function is correct but puzzling. Add a docstring that says what it returns, and **one** comment line (starting with `#`) above the `return` that explains **why** it adds 9 before dividing. The checks look for both.

In [ ]:
def pages_needed(lines, per_page=10):
    return (lines + per_page - 1) // per_page

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_doc_and_comment():\n    \"a docstring and a why-comment\"\n    assert pages_needed.__doc__, \"Add a docstring.\"\n    assert \"#\" in source, \"Add a comment explaining the + per_page - 1.\"\n    assert pages_needed(0) == 0 and pages_needed(1) == 1 and pages_needed(10) == 1 and pages_needed(11) == 2")

### Bug hunt: The missing docstring

`help(tidy)` shows no description, even though the function seems to have one. Fix it without changing what the function does.

In [ ]:
def tidy(text):
    text = " ".join(text.split())
    """Return text with runs of whitespace replaced by single spaces."""
    return text

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_doc():\n    \"the docstring is found\"\n    assert tidy.__doc__ and \"single spaces\" in tidy.__doc__, \"The docstring must be the first statement in the body.\"\n    assert tidy(\"  a   b \") == \"a b\"")

### Bug hunt: The docstring lies

`doctest` reports a failure in this function's examples. The function is right; one of the documented examples is wrong. Fix the docstring.

In [ ]:
def squares_sum(n):
    """Return 1 squared + 2 squared + ... + n squared.

    >>> squares_sum(3)
    14
    >>> squares_sum(4)
    20
    """
    total = 0
    for k in range(1, n + 1):
        total += k * k
    return total

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_doctest():\n    \"every example passes\"\n    import doctest\n    runner = doctest.DocTestRunner(verbose=False)\n    for t in doctest.DocTestFinder().find(squares_sum, \"squares_sum\", globs={\"squares_sum\": squares_sum}):\n        runner.run(t, out=lambda s: None)\n    assert runner.failures == 0, \"An example still doesn't match. Work out 1 + 4 + 9 + 16.\"\n    assert squares_sum(4) == 30, \"Don't change the function; it's correct.\"")

### Bug hunt: Quotes in the example

The example in this docstring fails, although the function is right and the words are right. Find the difference between what Python shows and what the docstring says.

In [ ]:
def shout(text):
    """Return text in capitals with three exclamation marks.

    >>> shout("hi")
    HI!!!
    """
    return text.upper() + "!!!"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_doctest():\n    \"the example passes\"\n    import doctest\n    runner = doctest.DocTestRunner(verbose=False)\n    for t in doctest.DocTestFinder().find(shout, \"shout\", globs={\"shout\": shout}):\n        runner.run(t, out=lambda s: None)\n    assert runner.failures == 0, \"Strings returned at the >>> prompt are shown with their quotes.\"")

### Puzzle: A documented function

Arrange the lines into a function with a docstring in the right place. One line is a trap.

(Put these lines in order on the website.)

### Exercise: What do they print?

Work out what each line prints.

```python
def a():
    "Say hello."
    return "hello"

def b():
    x = 1
    "Not a docstring."
    return x

print(a.__doc__)    # line 1
print(b.__doc__)    # line 2
print(a())          # line 3
```

In [ ]:
line_1 = "___"
line_2 = "___"
line_3 = "___"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"each line\"\n    assert line_1 == \"Say hello.\", \"A docstring can use single quotes too, if it's first.\"\n    assert line_2 == \"None\", \"In b, the string isn't the first statement.\"\n    assert line_3 == \"hello\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a documented toolkit

Write three text functions. Each must have a docstring with a summary line and **at least two doctest examples**, and every example must pass:

- `word_count(text)` returns the number of words (separated by any whitespace).
- `title_case(text)` returns the text with each word's first letter in capitals and the rest in lower case, words joined by single spaces, without using `.title()` or `.capitalize()`.
- `is_pangram(text)` returns `True` if the text contains every letter a to z, ignoring case.

In [ ]:
def word_count(text):
    ___

def title_case(text):
    ___

def is_pangram(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import doctest\ndef run_doc(f):\n    runner = doctest.DocTestRunner(verbose=False)\n    tests = doctest.DocTestFinder().find(f, f.__name__, globs={f.__name__: f})\n    n = 0\n    for t in tests:\n        n += len(t.examples)\n        runner.run(t, out=lambda s: None)\n    return n, runner.failures\ndef test_docs():\n    \"every function has at least two passing examples\"\n    for f in [word_count, title_case, is_pangram]:\n        assert f.__doc__, f.__name__ + \" needs a docstring.\"\n        n, fails = run_doc(f)\n        assert n >= 2, f.__name__ + \" needs at least two >>> examples.\"\n        assert fails == 0, f.__name__ + \": one of its examples doesn't match what it returns.\"\ndef test_behaviour():\n    \"the functions work\"\n    assert word_count(\"  the quick   brown fox \") == 4 and word_count(\"\") == 0\n    assert title_case(\"tHE   QUICK brown\") == \"The Quick Brown\"\n    assert is_pangram(\"The quick brown fox jumps over the lazy dog\") is True\n    assert is_pangram(\"hello\") is False\n    assert \".title(\" not in source and \".capitalize(\" not in source")

## Warm-up

You can work out what `f` does by reading its body, because it's one line. But what is `f` *for*? What should `a` be? Could `b` be negative? For a function with twenty lines, or one written by someone else, you need to be told.

## What a docstring is

A **docstring** is a string that comes **first** in a function's body. Python stores it with the function, and anyone can read it:

`help(power)` shows the function's name, its parameters and its docstring: exactly what someone needs to use it without reading its code. Try `help(len)`, `help(print)` or `help(str.split)` in the playground: every built-in function has a docstring.

The docstring must be the **first statement** in the body. Anywhere else, it's just an unused string:

`None`: Python didn't treat it as a docstring.

## Writing a good docstring

Python's style guide for docstrings, PEP 257, gives a simple shape:

1. **A one-line summary**, on the first line, ending with a full stop. Write it as a command: "Return the area...", not "Returns the area..." or "This function...".
2. If more is needed, **a blank line**, then the details: each parameter, what's returned, and anything surprising.
3. Optionally, **examples** of calls and their results.

For a short, obvious function, the summary line is enough:

```python
def is_even(n):
    """Return True if n is even."""
    return n % 2 == 0
```

For anything with several parameters, or any choices the caller should know about, add details:

Notice what the docstring doesn't say: nothing about `abs` or f-strings. Those are details of how it works, which the caller doesn't need. If you change how it works, the docstring should still be true.

## Contracts

A docstring describes a **contract** between the function and its caller:

- **What the caller must provide.** "n must be a whole number of at least 0." These are the function's **preconditions**.
- **What the function promises in return.** "Returns the n-th Fibonacci number." These are its **postconditions**.

If the caller keeps their side, the function must keep its side. If the caller breaks the contract, say by passing a negative number where the docstring asks for a positive one, the function doesn't have to behave sensibly. (Next week you'll learn to make a function check its preconditions and complain clearly.)

Writing the contract down before you write the body is a powerful habit. It forces you to decide exactly what the function is for, and it tells you which cases to test.

## Examples that check themselves

Examples are often the clearest part of a docstring. Python has a standard way to write them, copied from what the interactive shell looks like: `>>>` before the call, and the result on the next line.

```python
def initials(name):
    """Return the initials of a name, in capitals with full stops.

    >>> initials("ada lovelace")
    'A.L.'
    >>> initials("Grace Brewster Hopper")
    'G.B.H.'
    """
```

The result line is exactly what Python would show for that value: strings appear with their quotes, as `repr` writes them.

The clever part: Python's `doctest` module can **run** these examples and check that the results match. If they don't, it tells you:

When every example passes, `doctest` prints nothing (and the `print` around it shows `None`). Now break the function, for instance remove `.upper()`, and run it again: `doctest` shows the failing example, what was expected, and what it got.

In a normal Python file you'd usually write `doctest.testmod()` at the bottom, which checks every docstring in the file at once.

## Comments versus docstrings

Both are text for people, but they have different jobs:

| | Docstring | Comment |
|---|---|---|
| Starts with | triple quotes, first in the body | `#`, anywhere |
| Explains | **what** the function does and how to use it | **why** a piece of code is written the way it is |
| Read by | users of the function, and `help()` | people reading or changing the code |

The comment explains a surprising rule to someone reading the code. The docstring tells a user what they get.

## Practice

## Quiz

## Daily challenge

## Before you go

Choose a function you wrote this week and give it a full docstring: a summary, its contract, and two examples. Run its examples with `doctest`. Tomorrow is the Forge: a program built entirely from small, documented functions, plus the weekly quiz.